# Milestone 1 - Tool Calling Compatibility

Notebook này kiểm tra khả năng gọi function/tool của các model qua LiteLLM:

- `local-gemma`: Gemma4 trên Máy 1
- `mimo-pro`: Xiaomi MiMo 2.5 Pro
- `openai-model`: OpenAI GPT-5.4 mini
- `grok-model`: Grok 4.20 Reasoning qua Azure

Tool trong bài test là **tool giả**, không đọc hoặc ghi filesystem. Model chỉ cần tạo đúng `tool_calls` theo schema. Kết quả được lưu vào thư mục `results/` để so sánh.

In [ ]:
# ============================================================
# Cell 1: Cài dependency cho notebook và fixture test
# ============================================================
%pip install -q requests python-dotenv pytest

In [ ]:
# ============================================================
# Cell 2: Đọc cấu hình LiteLLM từ .env
# ============================================================
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv


def find_repo_env() -> Path:
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / ".env"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy file .env của repo")


ENV_PATH = find_repo_env()
REPO_ROOT = ENV_PATH.parent
TEST_ROOT = REPO_ROOT / "Notebooks" / "Test_Code_Editor"
RESULTS_DIR = TEST_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH, override=False)

LITELLM_URL = os.getenv("LITELLM_URL", "http://localhost:4000/v1").rstrip("/")
LITELLM_MASTER_KEY = os.getenv("LITELLM_MASTER_KEY", "sk-local")

MODEL_CASES = {
    "local": "local-gemma",
    "mimo": "mimo-pro",
    "openai": "openai-model",
    "grok": "grok-model",
}

# Đặt lớn hơn 1 khi cần đánh giá độ ổn định. Lần đầu nên dùng 1 để hạn chế chi phí.
RUNS_PER_MODEL = 1

print(f"Repo:       {REPO_ROOT}")
print(f"LiteLLM:    {LITELLM_URL}")
print(f"Kết quả:    {RESULTS_DIR}")
print(f"Số lần/model: {RUNS_PER_MODEL}")

In [ ]:
# ============================================================
# Cell 3: Kiểm tra LiteLLM và các model alias
# ============================================================
HEADERS = {
    "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
    "Content-Type": "application/json",
}

models_response = requests.get(
    f"{LITELLM_URL}/models",
    headers=HEADERS,
    timeout=30,
)
models_response.raise_for_status()
available_models = {
    item.get("id") for item in models_response.json().get("data", [])
}

print("Các model LiteLLM đang expose:")
for model_id in sorted(available_models):
    print(f"  - {model_id}")

missing_models = set(MODEL_CASES.values()) - available_models
if missing_models:
    raise RuntimeError(f"Thiếu model alias: {sorted(missing_models)}")

print("\n✅ Đủ các model alias cho Milestone 1")

In [ ]:
# ============================================================
# Cell 4: Tool schema giả và prompt dùng chung
# ============================================================
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_file_metadata",
            "description": "Get metadata for a file in the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {
                        "type": "string",
                        "description": "Workspace-relative path to the file.",
                    }
                },
                "required": ["file_path"],
                "additionalProperties": False,
            },
        },
    }
]

SYSTEM_PROMPT = (
    "Bạn đang được kiểm tra khả năng gọi công cụ. "
    "Khi người dùng yêu cầu kiểm tra file, bắt buộc gọi tool phù hợp. "
    "Không tự đoán metadata và không tuyên bố kết quả trước khi tool trả lời."
)

USER_PROMPT = (
    "Hãy dùng tool get_file_metadata để kiểm tra file calculator.py. "
    "Không tự đoán kết quả."
)

print(json.dumps(TOOLS, ensure_ascii=False, indent=2))
print(f"\nPrompt: {USER_PROMPT}")

In [ ]:
# ============================================================
# Cell 5: Gọi model và chấm tool call
# ============================================================

def parse_tool_arguments(raw_arguments):
    if isinstance(raw_arguments, dict):
        return raw_arguments, True, None
    try:
        return json.loads(raw_arguments or "{}"), True, None
    except (TypeError, json.JSONDecodeError) as error:
        return None, False, str(error)


def call_tool_test(model_alias: str, timeout: int = 180) -> dict:
    payload = {
        "model": model_alias,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ],
        "tools": TOOLS,
        "tool_choice": "auto",
        "temperature": 0,
        "max_tokens": 300,
    }

    started = time.perf_counter()
    try:
        response = requests.post(
            f"{LITELLM_URL}/chat/completions",
            headers=HEADERS,
            json=payload,
            timeout=timeout,
        )
        latency = time.perf_counter() - started
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as error:
        latency = time.perf_counter() - started
        error_response = getattr(error, "response", None)
        return {
            "passed": False,
            "requested_model": model_alias,
            "http_status": error_response.status_code if error_response is not None else None,
            "latency_seconds": round(latency, 3),
            "error": error_response.text[:3000] if error_response is not None else str(error),
        }

    choices = data.get("choices", [])
    choice = choices[0] if choices else {}
    message = choice.get("message", {}) or {}
    tool_calls = message.get("tool_calls") or []
    first_call = tool_calls[0] if tool_calls else {}
    function = first_call.get("function", {}) or {}
    arguments, arguments_valid, arguments_error = parse_tool_arguments(
        function.get("arguments")
    )

    tool_name = function.get("name")
    file_path = arguments.get("file_path") if isinstance(arguments, dict) else None
    content = message.get("content") or ""

    checks = {
        "has_tool_calls": bool(tool_calls),
        "exactly_one_tool_call": len(tool_calls) == 1,
        "correct_tool_name": tool_name == "get_file_metadata",
        "arguments_are_valid_json": arguments_valid,
        "has_file_path": isinstance(file_path, str) and bool(file_path.strip()),
        "targets_calculator_py": isinstance(file_path, str)
        and Path(file_path).name == "calculator.py",
    }

    return {
        "passed": all(checks.values()),
        "requested_model": model_alias,
        "actual_model": data.get("model"),
        "http_status": response.status_code,
        "latency_seconds": round(latency, 3),
        "finish_reason": choice.get("finish_reason"),
        "assistant_content": content,
        "tool_call_count": len(tool_calls),
        "tool_name": tool_name,
        "tool_arguments_raw": function.get("arguments"),
        "tool_arguments": arguments,
        "arguments_error": arguments_error,
        "checks": checks,
        "usage": data.get("usage", {}),
    }


def print_test_result(label: str, result: dict):
    status = "PASS" if result.get("passed") else "FAIL"
    print(f"\n{'=' * 22} {label.upper()} - {status} {'=' * 22}")
    print(f"Requested model: {result.get('requested_model')}")
    print(f"Actual model:    {result.get('actual_model')}")
    print(f"HTTP status:     {result.get('http_status')}")
    print(f"Latency:         {result.get('latency_seconds')}s")
    if result.get("error"):
        print(f"Error:           {result['error']}")
        return
    print(f"Finish reason:   {result.get('finish_reason')}")
    print(f"Tool name:       {result.get('tool_name')}")
    print(f"Arguments:       {result.get('tool_arguments')}")
    print(f"Assistant text:  {result.get('assistant_content')!r}")
    print(f"Usage:           {result.get('usage')}")
    print("Checks:")
    for check_name, passed in result.get("checks", {}).items():
        print(f"  [{'OK' if passed else 'FAIL'}] {check_name}")

In [ ]:
# ============================================================
# Cell 6: Chạy cùng một bài test trên cả ba model
# Cell này phát sinh request tới Cloud APIs khi test MiMo/OpenAI.
# ============================================================
all_results = {}

for label, model_alias in MODEL_CASES.items():
    model_runs = []
    for run_number in range(1, RUNS_PER_MODEL + 1):
        print(f"\nĐang test {label} ({model_alias}), lần {run_number}/{RUNS_PER_MODEL}...")
        result = call_tool_test(model_alias)
        result["run_number"] = run_number
        result["tested_at"] = datetime.now(timezone.utc).isoformat()
        model_runs.append(result)
        print_test_result(label, result)
    all_results[label] = model_runs

In [ ]:
# ============================================================
# Cell 7: Lưu JSON và in bảng tổng kết
# ============================================================
summary = []

for label, runs in all_results.items():
    output_path = RESULTS_DIR / f"{label}.json"
    output_path.write_text(
        json.dumps(runs, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    passed_runs = sum(bool(run.get("passed")) for run in runs)
    average_latency = (
        sum(float(run.get("latency_seconds", 0)) for run in runs) / len(runs)
        if runs
        else 0
    )
    summary.append(
        {
            "provider": label,
            "model": MODEL_CASES[label],
            "passed_runs": passed_runs,
            "total_runs": len(runs),
            "pass_rate": passed_runs / len(runs) if runs else 0,
            "average_latency_seconds": round(average_latency, 3),
        }
    )
    print(f"Đã lưu: {output_path}")

(RESULTS_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 78)
print(f"{'Provider':<12} {'Model':<18} {'Passed':>10} {'Pass rate':>12} {'Avg latency':>14}")
print("-" * 78)
for item in summary:
    print(
        f"{item['provider']:<12} "
        f"{item['model']:<18} "
        f"{item['passed_runs']:>4}/{item['total_runs']:<5} "
        f"{item['pass_rate']:>11.0%} "
        f"{item['average_latency_seconds']:>12.3f}s"
    )

print("\nĐiều kiện đi tiếp: model phải PASS tool name, JSON arguments và file_path.")
print("Để đánh giá độ ổn định trước khi cấp quyền ghi, đặt RUNS_PER_MODEL = 10.")

## Cách diễn giải kết quả

- **PASS**: model đủ điều kiện thử agent `read_only` ở giai đoạn tiếp theo.
- **FAIL vì không có `tool_calls`**: model chỉ trả lời text; chưa được cấp filesystem tool.
- **FAIL vì JSON arguments**: cần điều chỉnh prompt/schema hoặc không dùng model cho agent tự động.
- **HTTP 4xx**: provider/model có thể không hỗ trợ tool calling theo API hiện tại.
- **HTTP 5xx/timeout**: kiểm tra upstream, ngrok, LiteLLM và thử lại trước khi kết luận.

Một lần PASS chỉ chứng minh tính tương thích ban đầu. Trước khi cấp quyền ghi file, chạy ít nhất 10 lần và yêu cầu tỷ lệ đúng từ 90% trở lên.